#### Agent loop

- Declare the functions and tool dictionaries
- Handle tool calls function
    - to call the tools and returns the results of the tools as messages
- Agent loop 

In [1]:
def get_city_temperature(city: str):

    if city.lower() == "bangalore":
        temperature = 25
    else:
        temperature = 50
    return temperature

get_city_temperature_json = {
    "name": "get_city_temperature",
    "description": "Use this tool to get the current temperature of any city in the world",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": get_city_temperature_json}]

In [2]:
import json

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}, with arguments {arguments}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def chat(message, history):
    system_prompt = "You are a helpful assistant"
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        # We can add a for loop here to limit number of llm calls
        # This is the call to the LLM - see that we pass in the tools json

        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [7]:
message = "What is the temperature in Delhi"

answer = chat(message=message, history=[])
print(answer)

Tool called: get_city_temperature, with arguments {'city': 'Delhi'}
The current temperature in Delhi is 50°C.


##### Try out gradio

In [11]:
import gradio as gr

In [ ]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: get_city_temperature, with arguments {'city': 'Bangalore'}
Tool called: get_city_temperature, with arguments {'city': 'Delhi'}
